In [2]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

import os
import sys
import pandas as pd
import numpy as np
from sklearn.preprocessing import RobustScaler, PowerTransformer, MinMaxScaler
from sklearn.pipeline import make_pipeline

# ========= ここを環境に合わせて指定してください =========
DIR_PATH   = "/data2/ssk/githubtest/sim2real/data/foldX/"       # CSVがあるディレクトリ
FILE_NAME  = "/data2/ssk/githubtest/sim2real/data/foldX/4idl_all-var_ddg_with_rosettaddg_with_foldx.csv"         # 入力CSVファイル名
COL1       = "ddg"       # 元の数値カラム名（マイナスを掛ける対象）

# 省略可（未指定なら自動命名）
COL2_NAME  = None                 # 新カラム2名（例: "target_value_neg"）
COL3_NAME  = None                 # 新カラム3名（例: "target_value_scaled01"）
OUT_NAME   = None                 # 出力CSVファイル名（例: "sample_processed.csv"）
# ================================================


def main():
    in_path = os.path.join(DIR_PATH, FILE_NAME)
    if not os.path.isfile(in_path):
        print(f"入力ファイルが見つかりません: {in_path}", file=sys.stderr)
        sys.exit(1)

    df = pd.read_csv(in_path)

    if COL1 not in df.columns:
        print(f"指定カラムが見つかりません: {COL1}", file=sys.stderr)
        sys.exit(1)

    col2 = COL2_NAME if COL2_NAME else f"{COL1}_neg"
    col3 = COL3_NAME if COL3_NAME else f"{COL1}_scaled01"

    # 数値化（非数は NaN）
    col1_numeric = pd.to_numeric(df[COL1], errors="coerce")

    # 新カラム2：マイナスを掛ける
    df[col2] = -col1_numeric

    # スケーリング前の欠損対応：fit用に中央値で補完（出力は元NaNを維持）
    x = df[col2]
    na_mask = x.isna()
    if na_mask.all():
        print(f"{col2} がすべて NaN のためスケーリングできません。", file=sys.stderr)
        sys.exit(1)
    x_filled = x.fillna(x.median())

    # パイプライン：RobustScaler → PowerTransformer(yeo-johnson, standardize=True) → MinMaxScaler(0,1)
    pipe = make_pipeline(
        RobustScaler(),
        PowerTransformer(method='yeo-johnson', standardize=True),
        MinMaxScaler(feature_range=(0, 1))
    )
    pipe.fit(x_filled.to_numpy().reshape(-1, 1))
    scaled = pipe.transform(x_filled.to_numpy().reshape(-1, 1)).flatten()

    # 元がNaNだった場所はNaNに戻す
    scaled_series = pd.Series(scaled, index=df.index)
    scaled_series[na_mask] = np.nan
    df[col3] = scaled_series

    # 出力ファイル名
    if OUT_NAME:
        out_file = OUT_NAME
    else:
        stem, ext = os.path.splitext(FILE_NAME)
        out_file = f"{stem}_processed.csv"
    out_path = os.path.join(DIR_PATH, out_file)

    df.to_csv(out_path, index=False, encoding="utf-8-sig")
    print(f"保存しました: {out_path}")
    print(f"追加カラム: {col2}, {col3}")

if __name__ == "__main__":
    main()


保存しました: /data2/ssk/githubtest/sim2real/data/foldX/4idl_all-var_ddg_with_rosettaddg_with_foldx_processed.csv
追加カラム: ddg_neg, ddg_scaled01
